# Signal card: the first screen for a signal

Before any trading rule or backtest: does the signal predict the return that follows, and could that be luck, a leak or one lucky stretch?
One call gives a card with the same checks for every idea (`src/research/signal_checks.py`; notes in `docs/signal_checks_plan.md`).

Where it sits: idea and pre-registration → **signal card** → strategy test with costs → one look at the frozen window → paper.
"No evidence" stops a signal here. The pass marks are in `research/signal_standard.toml` and are not chosen per signal.

This notebook passes `ledger=False` so that running the examples adds nothing to the trial ledger. **Leave that out in real research:** a card is a look at the data and counts as a trial.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from src.research import signal_checks as sc, signal_lab

wide = signal_lab.load_universe(top_n=50, kraken_only=True, listing="point_in_time")   # daily closes, volume, taker flow, funding per coin
feats = signal_lab.features(wide)
inside = wide["universe"].astype(bool)
close = wide["close"].where(inside)
sc.load_standard()

## 1. One market

`signal` is what is known at each bar's close, `close` the prices on the same index. Each horizon is a trial; the verdict is for the first one, chosen before looking.
`direction=+1` says a high signal should be followed by a high return (leave it out if you have no view, and the tests are two-sided).

In [ ]:
btc = wide["close"]["BTCUSDT"].dropna()
signal = -wide["funding"]["BTCUSDT"].rolling(7).mean().reindex(btc.index)      # low funding: longs are not crowded
card = sc.signal_card(signal, btc, horizons=[7, 1, 30], name="BTC minus 7-day funding", direction=1, ledger=False)
print(card.verdict, "-", card.reason)
card.table

In [ ]:
card.by_horizon.round(3)      # the same numbers at every horizon asked for

In [ ]:
card.plot();

How to read it:

- **IC and its interval.** The rank correlation with the return that followed. The interval uses the *independent* observations, which for a slow signal against overlapping returns is far fewer than the bars.
- **Placebo p.** The same signal against returns from other dates. If many shifts do as well, the result is what luck looks like for a series this persistent.
- **IC kept 1 bar late.** A real effect survives acting one bar later; one that needs the signal's own bar does not.
- **Blocks.** The sign in each calendar year, and the IC without the best year.
- **Best bucket.** The forward return of the best fifth of bars against the average, next to one round trip. A real but tiny effect fails here.
- **Verdict.** *Worth a strategy test* (every check that applies passes), *weak* (beats luck but fails something), *no evidence*, or *short history* (too few independent observations to say).

## 2. A ranking of markets

Pass DataFrames of dates by markets. The card then scores each date's ranking of markets against their returns over the next `horizon` bars, relative to that date's average.

In [ ]:
ranking = feats["taker_buy_share_7d"].where(inside)
card = sc.signal_card(ranking, close, horizons=[7], name="taker-buy share, 7 days", direction=1, ledger=False)
print(card.verdict, "-", card.reason)
card.table

This one is instructive. By dates the IC is clearly positive, but the placebo's average is as large as the IC itself, and the signal measured against each coin's own history predicts nothing.
The ranking favours the same coins throughout and those coins rose more: a fixed tilt between about thirty coins, which is far less evidence than 1,400 dates suggest.

In [ ]:
per = card.details["per_market"]                                     # each market's own time-series IC, for reference
per[per["independent observations"] >= 30].sort_values("IC").round(3).tail(8)

## 3. Several candidates at once

With many candidates the best one always looks good. `compare` gives each a card (each costs a trial), groups the ones that are the same idea, and gives the IC the best of the bunch reaches by luck.

In [ ]:
candidates = {"minus funding, 7d": -feats["funding_7d"].where(inside), "taker-buy share, 7d": feats["taker_buy_share_7d"].where(inside),
              "30-day return": feats["mom_30d"].where(inside), "7-day return": np.log(close).diff(7), "8-day return": np.log(close).diff(8)}
board = sc.compare(candidates, close, horizon=7, ledger=False)
print("luck threshold:", round(board.luck_threshold, 3), "| the same idea twice:", board.redundant)
board.table

## 4. When the signal is fitted: rerun its code

A finished series cannot show whether its code leaks. `pipeline_checks` takes the function that builds the signal and
(1) rebuilds it on prices whose returns were put in a random order, where there is nothing to predict, and (2) rebuilds it on the first 50%, 70% and 90% of the data to see whether later data change earlier values.
A plain feature passes both trivially. Use it for anything with a fit in it: forecasts, regimes, conditional scores.

In [ ]:
def clean(prices):                       # 20-day return: uses the past only
    return np.log(prices).diff(20)

def standardised(prices):                # the same, scaled by the mean and spread of the WHOLE sample: hindsight
    change = np.log(prices).diff(20)
    return (change - change.mean()) / change.std()

def leaky(prices):                       # knows tomorrow's return
    return np.log(prices).diff().shift(-1) + 0.01 * np.log(prices).diff(20)

for build in (clean, standardised, leaky):
    result = sc.pipeline_checks(build, btc, horizon=1, runs=10)
    print(f"{build.__name__:13}", result)
result.table

## 5. An event with its own forward return

When the return that follows is not a fixed number of bars (the afternoon after a morning, the week after a Friday), pass it as `forward=`.
The checks that need prices (acting a bar late, the trailing-return baseline) are then marked as not applicable.

```python
card = sc.signal_card(morning_return, forward=afternoon_return, name="H10 morning move, BTC", direction=1, round_trip_bps=11)
```

## 6. In a study

```python
card = sc.signal_card(signal, close, horizons=[7], name="my_signal", family="H12")   # logged to the trial ledger
card.save()                                                                         # data/research/signal_cards/<name>_<time>/
sc.require_card("my_signal", "H12")   # at the top of the strategy test: raises on "no evidence" unless the
                                      # pre-registration has a line  SIGNAL CARD OVERRIDE: <reason>
```

What the card is not: proof. A few noise signals in a hundred still get the top verdict (the tests are at the 5% level), which is why every card costs a trial and why a strategy test, the frozen window and paper come after it.